<a href="https://colab.research.google.com/github/andrewxu319/cuda-fft/blob/main/CUDA_FFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Accidentally reinvented the Stockham FFT after trying to do Cooley-Tukey in-place and without recursion.

In [1]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmp0xr_7o4n".


In [14]:
%%cuda

// to do: grid sizes, optimizations using shared memory

#include <stdio.h>
#include <cmath>
#include <complex>
#include <vector>
#include <fstream>
#include <cassert>
#include <cuda_runtime.h>
#include <cuda/std/complex>

constexpr size_t MAX_BLOCK_SIZE = 1024;
using complex = std::complex<double>;
using cu_complex = cuda::std::complex<double>;
constexpr double pi = 3.14159265358979323846;

__device__ void fft_small_kernel_device(cu_complex* full_data_array, size_t n) {
    // if large matrix, fft each row
    // here, n means the size of the input for THIS PARTICULAR BLOCK
    cu_complex* data = &full_data_array[blockIdx.x * blockDim.x]; // pointer to start of the current row
    size_t i{ threadIdx.x };

    for (size_t stride{n / 2}; stride > 0; stride /= 2) {
        // note that j resets after i=n/2
        size_t j{ (i % (n/2)) / stride };
        size_t subprob_offset{ i % stride };
        cu_complex y_e_j;
        cu_complex y_o_j;
        y_e_j = data[2 * j * stride + subprob_offset];
        y_o_j = data[(2 * j + 1) * stride + subprob_offset];
        __syncthreads();
        double theta{ -2.0 * pi / (n / stride) * j };
        double real, imag;
        sincos(theta, &imag, &real);
        cu_complex omega_to_j{ real, imag };
        data[i] = y_e_j + (-1 + 2 * (static_cast<double>(i < n / 2))) * omega_to_j * y_o_j; // avoids branching. equivalent to (i < n / 2) ? 1 : -1
        __syncthreads();
    }
}

__global__ void fft_small_kernel(cu_complex* data, size_t n) {
    fft_small_kernel_device(data, n);
}

__device__ void correction_multiplication_kernel(cu_complex* data, size_t n, size_t cols) {
    size_t row{ blockIdx.x };
    size_t col{ threadIdx.x };
    double theta{ -2.0 * pi * row * col / n };
    double real, imag;
    sincos(theta, &imag, &real);
    data[row * cols + col] *= cu_complex{ real, imag };
}

__global__ void transpose_kernel(cu_complex* dest, cu_complex* src, size_t n, size_t rows, size_t cols, size_t TILE_DIM, size_t BLOCK_ROWS) {
    // each thread is responsible for several vertically evenly spaced apart entries in the same column
    extern __shared__ cu_complex tile[]; // original, untransposed

    // location in the whole matrix
    size_t row{ blockIdx.y * TILE_DIM + threadIdx.y };
    size_t col{ blockIdx.x * TILE_DIM + threadIdx.x };

    for (size_t i{}; i < TILE_DIM; i += BLOCK_ROWS) { // i is the index of the start of each of this tile's row relative
        tile[(threadIdx.y + i) * (TILE_DIM + 1) + threadIdx.x] = src[(row + i) * cols + col];
    }

    __syncthreads();

    size_t row_tr{ blockIdx.x * TILE_DIM + threadIdx.y };
    size_t col_tr{ blockIdx.y * TILE_DIM + threadIdx.x };
    for (size_t i{}; i < TILE_DIM; i += BLOCK_ROWS) {
        dest[(row_tr + i) * rows + col_tr] = tile[(threadIdx.x) * (TILE_DIM + 1) + (threadIdx.y + i)];
    }
}

__global__ void bailey_helper_kernel(cu_complex* data, size_t n, size_t rows, size_t cols) {
    size_t row{ blockIdx.x };
    size_t col{ threadIdx.x };
    fft_small_kernel_device(data, cols);
    __syncthreads();
    correction_multiplication_kernel(data, n, cols);
}

void fft_cuda(cu_complex* h_data, size_t n) {
    cu_complex* d_data;
    cudaMalloc((void**)&d_data, n * sizeof(cu_complex));

    if (n <= MAX_BLOCK_SIZE) {
        cudaMemcpy(d_data, h_data, n * sizeof(cu_complex), cudaMemcpyHostToDevice);;
        fft_small_kernel<<<1, n>>>(d_data, n);
        cudaMemcpy(h_data, d_data, n * sizeof(cu_complex), cudaMemcpyDeviceToHost);
        cudaDeviceSynchronize();
    } else if (n <= MAX_BLOCK_SIZE * MAX_BLOCK_SIZE) { // if possible to fft a whole row & a whole column in one go
        cudaMemcpy(d_data, h_data, n * sizeof(cu_complex), cudaMemcpyHostToDevice);

        // remember n still has to be a power of 2
        size_t cols{ 64 };
        size_t rows{ 32 };
        // size_t cols{ MAX_BLOCK_SIZE }; // change this to make sure cols and rows are both divisible by 32
        // size_t rows{ n / cols };
        size_t blocks_per_grid{ (n + cols - 1) / cols }; // optimize more

        cu_complex* d_transposed;
        cudaMalloc((void**)&d_transposed, n * sizeof(cu_complex));

        // step 1 fft each row
        // step 2 multiply by correction factor
        bailey_helper_kernel<<<blocks_per_grid, cols>>>(d_data, n, rows, cols); // each block is a row

        // step 3 transpose
        size_t TILE_DIM{ 32 };
        size_t BLOCK_ROWS{ 8 };
        dim3 grid_dim{ (cols + TILE_DIM - 1) / TILE_DIM, (rows + TILE_DIM - 1) / TILE_DIM, 1 };
        dim3 block_dim{ TILE_DIM, BLOCK_ROWS, 1 }; // CHANGE
        size_t shared_mem_size{ TILE_DIM * (TILE_DIM + 1) * sizeof(cu_complex) }; // +1 for bank conflicts
        transpose_kernel<<<grid_dim, block_dim, shared_mem_size>>>(d_transposed, d_data, n, rows, cols, TILE_DIM, BLOCK_ROWS);

        // step 4 fft each row of transposed matrix
        size_t blocks_per_grid_transposed{ (n + rows - 1) / rows };
        fft_small_kernel<<<blocks_per_grid_transposed, rows>>>(d_transposed, rows);

        // step 5 transpose back
        dim3 grid_dim_transposed{ (rows + TILE_DIM - 1) / TILE_DIM, (cols + TILE_DIM - 1) / TILE_DIM, 1 };
        transpose_kernel<<<grid_dim_transposed, block_dim, shared_mem_size>>>(d_data, d_transposed, n, cols, rows, TILE_DIM, BLOCK_ROWS);

        cudaMemcpy(h_data, d_transposed, n * sizeof(cu_complex), cudaMemcpyDeviceToHost);
        cudaDeviceSynchronize();

        cudaFree(d_transposed);
    }

    cudaFree(d_data);
}

int main() {
    std::string line;
    std::ifstream file{ "input.txt" };
    std::vector<cu_complex> data_vector{};
    while (std::getline(file, line)) {
        data_vector.emplace_back(cu_complex{ std::stoi(line), 0.0 });
    }
    size_t n{ data_vector.size() };
    assert(n > 0 && ((n & (n - 1)) == 0)); // power of 2
    cu_complex* data = data_vector.data();

    fft_cuda(data, n);

    for (size_t i{}; i < n; i++) {
        printf("%d, %f", i, cuda::std::abs(data[i]));
        printf("\n");
    }
}

/////////////////////////////////////////////////////

void fft_ct_cpu(complex* data, size_t n) {
    if (n == 1) {
        return;
    }

    complex omega{ std::exp(complex{0, - 2 * pi / n}) };
    std::vector<complex> data_e(n/2);
    std::vector<complex> data_o(n/2);
    for (size_t i{}; i < n/2; i++) {
        data_e[i] = data[2 * i];
        data_o[i] = data[2 * i + 1];
    }
    fft_ct_cpu(data_e.data(), n/2);
    fft_ct_cpu(data_o.data(), n/2);

    complex omega_to_i{ 1 };
    for (size_t i{}; i < n/2; i++) {
        data[i] = data_e[i] + omega_to_i * data_o[i];
        data[i + n/2] = data_e[i] - omega_to_i * data_o[i];
        omega_to_i *= omega;
    }
}

void ifft_ct_cpu_recursion(complex* data, size_t n) {
    if (n == 1) {
        return;
    }

    complex omega{ std::exp(complex{0, 2 * pi / n}) };
    std::vector<complex> data_e(n/2);
    std::vector<complex> data_o(n/2);
    for (size_t i{}; i < n/2; i++) {
        data_e[i] = data[2 * i];
        data_o[i] = data[2 * i + 1];
    }
    ifft_ct_cpu_recursion(data_e.data(), n/2);
    ifft_ct_cpu_recursion(data_o.data(), n/2);

    complex omega_to_i{ 1 };
    for (size_t i{}; i < n/2; i++) {
        data[i] = (data_e[i] + omega_to_i * data_o[i]);
        data[i + n/2] = (data_e[i] - omega_to_i * data_o[i]) ;
        omega_to_i *= omega;
    }
}

void ifft_ct_cpu(complex* data, size_t n) {
    ifft_ct_cpu_recursion(data, n);
    double n_reciprocal{ 1 / static_cast<double>(n) };
    for (size_t i{}; i < n; i++) {
        data[i] *= n_reciprocal;
    }
}

0, 0.000000
1, 0.000000
2, 0.000000
3, 0.000000
4, 0.000000
5, 0.000000
6, 0.000000
7, 0.000000
8, 0.000000
9, 0.000000
10, 0.000000
11, 0.000000
12, 0.000000
13, 0.000000
14, 0.000000
15, 0.000000
16, 0.000000
17, 0.000000
18, 0.000000
19, 0.000000
20, 0.000000
21, 0.000000
22, 0.000000
23, 0.000000
24, 0.000000
25, 0.000000
26, 0.000000
27, 0.000000
28, 0.000000
29, 0.000000
30, 0.000000
31, 0.000000
32, 1303.797805
33, 20.091695
34, 10.173220
35, 6.855131
36, 5.206975
37, 4.230575
38, 3.591664
39, 3.146740
40, 2.824098
41, 2.584037
42, 2.402919
43, 2.265921
44, 2.163412
45, 2.089025
46, 2.038563
47, 2.009376
48, 2.000002
49, 2.009983
50, 2.039807
51, 2.090970
52, 2.166163
53, 2.269640
54, 2.407850
55, 2.590551
56, 2.832776
57, 3.158526
58, 3.608193
59, 4.254927
60, 5.245685
61, 6.924814
62, 10.331348
63, 20.727444
64, 0.000000
65, 0.000000
66, 0.000000
67, 0.000000
68, 0.000000
69, 0.000000
70, 0.000000
71, 0.000000
72, 0.000000
73, 0.000000
74, 0.000000
75, 0.000000
76, 0.000000
77